# Solar Filament Segmentation Challenge 2026 — public raw-JPEG inference

**Reproducibility notebook, experimental release (13 November 2026).** This notebook runs the released H9b model family from the organiser-supplied 180 unlabelled test JPEGs and ten SHA-256-pinned model weights. It produces a COCO-RLE CSV without loading private prediction caches, Kaggle TEST labels, or temporal-neighbour images.

The notebook contains all execution steps, from native U-Net H3 and Mask R-CNN inference to cross-view consensus, TRAIN-only no-site component selection, crop refinement and assembly. Work is stored on a large HDD, with at most one CUDA job on the designated GPU0; GPU1 is never used. The first two stages require the server's NVIDIA/PyTorch environment. The last stages use CPU FP32 and are independently restartable.

**Important scientific limitation:** the historically accepted H9b upload (submission 58112557, public PQ 0.39) used a separate earlier own-model control prediction for images without a surviving component. A clean raw-image pipeline cannot silently reuse that archived control CSV. This notebook instead uses a fully deterministic, publicly reproducible semantic-component fallback. As a result, its CSV may differ from the accepted original. Validate locally before considering any Kaggle submission. The notebook itself does **not** submit to Kaggle, and a reproducible output does not imply improved PQ.

The official test JPEGs are not redistributed. Provide the competition-authorized image directory through the environment variable SOLARFIL_TEST_IMAGES. Set SOLARFIL_REPLAY_OUT to an HDD directory with at least 20 GiB free, if the default path is unsuitable.

In [ ]:
from __future__ import annotations

import csv
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import pickle
import shutil
import subprocess
import sys
import time

import cv2
import numpy as np

ROOT = Path.cwd().resolve()
MODELS = ROOT / "release/models"
SOURCES = ROOT / "release/source"
SCRIPTS = ROOT / "scripts"
GPU_UUID = "GPU-b2046dc8-91f7-fd8a-b237-96bb9e293d14"
FOLDS = ("F1", "F2", "F3", "F4")
IMAGES = Path(os.environ["SOLARFIL_TEST_IMAGES"]).expanduser().resolve()
OUT = Path(os.environ.get(
    "SOLARFIL_REPLAY_OUT",
    "/srv/almacen/nas-2tb-1/solar-filament-2026/clean-h9b-replay-20261113"
)).expanduser().resolve()
if "/mnt/datos" in (str(OUT), *map(str, OUT.parents)):
    raise RuntimeError("Refusing to fill /mnt/datos; choose the research HDD.")
if not (ROOT / "release/model_metadata/WEIGHTS_SHA256SUMS.txt").exists():
    raise FileNotFoundError("Run from a complete public repository checkout.")
FILES = sorted(IMAGES.glob("*.jpeg"))
if len(FILES) != 180 or len({p.stem for p in FILES}) != 180:
    raise RuntimeError("Expected 180 distinct official competition TEST JPEGs.")
if shutil.disk_usage(OUT.parent if OUT.parent.exists() else ROOT).free < 20 * 1024**3:
    raise OSError("At least 20 GiB must be available on the selected HDD.")
OUT.mkdir(parents=True, exist_ok=True)
H3 = OUT / "h3"
DETECTORS = OUT / "detectors"
OVERLAY = OUT / "pipeline"
CROPS = OUT / "crops"
REFINERS = OUT / "refiners"
CROPS.mkdir(exist_ok=True)
REFINERS.mkdir(exist_ok=True)
cv2.setNumThreads(1)
sys.path.insert(0, str(SOURCES))
sys.path.insert(0, str(SCRIPTS))

def sha(path: Path) -> str:
    """Compute a SHA-256 of an arbitrary checkpoint or prediction."""
    value = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1048576), b""):
            value.update(block)
    return value.hexdigest()

def write_json_once(path: Path, payload: dict) -> None:
    """Atomically persist one immutable receipt, never overwrite a prior run."""
    if path.exists():
        raise FileExistsError(path)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("x", encoding="utf8") as stream:
        json.dump(payload, stream, indent=2, sort_keys=True)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)

def gpu_available_exclusively() -> None:
    """Verify UUID and CUDA process list immediately before each GPU command."""
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,uuid,utilization.gpu,memory.used",
         "--format=csv,noheader"], capture_output=True, text=True, check=True
    )
    print(gpu.stdout)
    if GPU_UUID not in gpu.stdout:
        raise RuntimeError("Assigned GPU0 is absent.")
    processes = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=gpu_uuid,pid",
         "--format=csv,noheader"], capture_output=True, text=True, check=True
    )
    if any(GPU_UUID in row for row in processes.stdout.splitlines()):
        raise RuntimeError("GPU0 already hosts CUDA. Leave existing processes alone.")

print("Official unlabeled images:", len(FILES), "HDD output:", OUT)

## 1. Verify all ten public model weights and execution dependencies

The exact public weights and original source files must be present in the checkout. The manifest is verified before inference. These assets are independently released, not downloaded as untrusted runtime code.

In [ ]:
manifest_file = ROOT / "release/model_metadata/WEIGHTS_SHA256SUMS.txt"
manifest = {}
for line in manifest_file.read_text(encoding="utf8").splitlines():
    expected, basename = line.split()
    if len(expected) != 64 or basename in manifest:
        raise RuntimeError("Malformed or duplicate weight signature.")
    weight = MODELS / basename
    if not weight.is_file() or sha(weight) != expected:
        raise RuntimeError(f"Published checkpoint failed SHA-256: {basename}")
    manifest[basename] = expected
assert len(manifest) == 10, "The published H9b ensemble has exactly ten checkpoints."
selector_json = ROOT / "release/model_metadata/h9b_nosite_selector.json"
selector_path = ROOT / "release/model_metadata/h9b_nosite_selector.pkl"
selector_meta = json.loads(selector_json.read_text(encoding="utf8"))
if selector_meta["weight_sha256"] != sha(selector_path):
    raise RuntimeError("Frozen TRAIN-only selector integrity failure.")
if not selector_meta["no_competition_test_annotations"]:
    raise RuntimeError("TRAIN-only feature provenance was not certified.")
subprocess.run([sys.executable, str(SCRIPTS / "verify_release_assets.py")],
               check=True, cwd=ROOT)
print("MODEL_SHA256_VERIFIED", len(manifest), "selector", sha(selector_path))

## 2. Generate all four native-resolution U-Net H3 probability maps

Each fold runs separately with GPU0 isolated by its physical UUID. Checkpointed PNGs and per-fold receipts allow an interrupted run to resume. A completed receipt prevents redundant inference.

In [ ]:
h3_env = os.environ.copy()
h3_env["CUDA_VISIBLE_DEVICES"] = GPU_UUID
h3_env["OMP_NUM_THREADS"] = "2"
for fold in FOLDS:
    target = H3 / fold
    receipt = H3 / f"{fold}_h3_replay_receipt.json"
    expected = {p.stem for p in FILES}
    if receipt.exists():
        evidence = json.loads(receipt.read_text(encoding="utf8"))
        if evidence.get("images_processed") != 180 or not evidence.get("full_180_images"):
            raise RuntimeError(f"Incomplete saved H3 run: {fold}")
        if {p.stem for p in target.glob("*.png")} != expected:
            raise RuntimeError(f"H3 file inventory changed: {fold}")
        print("H3_REUSED_VERIFIED", fold)
        continue
    gpu_available_exclusively()
    command = [
        sys.executable, "-u", str(SCRIPTS / "replay_h3_from_original_images.py"),
        "--images", str(IMAGES), "--out", str(H3), "--fold", fold,
        "--device", "cuda", "--gpu-uuid", GPU_UUID,
    ]
    if target.exists() and any(target.glob("*.png")):
        command.append("--resume")
    subprocess.run(command, cwd=ROOT, env=h3_env, check=True)
    if not receipt.exists():
        raise RuntimeError(f"No H3 completion receipt: {fold}")
print("FOUR_H3_FOLDS_READY")

## 3. Execute all four public Mask R-CNN detectors

The detector reconstruction produces two geometric views per fold and image. This is a **sequential** GPU job, never concurrent with H3 inference. The public detector source checks hardware occupancy before CUDA initialization.

In [ ]:
detector_receipt = DETECTORS / "receipt.json"
if detector_receipt.exists():
    detector_meta = json.loads(detector_receipt.read_text(encoding="utf8"))
    if not all(len(list((DETECTORS / fold).glob("*.pkl"))) == 180 for fold in FOLDS):
        raise RuntimeError("Saved detector stage is incomplete.")
    print("DETECTORS_REUSED_VERIFIED", detector_meta.get("status"))
else:
    gpu_available_exclusively()
    detector_env = os.environ.copy()
    detector_env["CUDA_VISIBLE_DEVICES"] = GPU_UUID
    detector_env["OMP_NUM_THREADS"] = "2"
    detector_env["PYTHONPATH"] = os.pathsep.join([str(SOURCES), str(ROOT)])
    subprocess.run([
        sys.executable, "-u",
        str(SCRIPTS / "p82_full_public_detector_replay.py"),
        "--release", str(ROOT / "release"),
        "--images", str(IMAGES),
        "--out", str(DETECTORS),
    ], cwd=ROOT, env=detector_env, check=True)
    if not detector_receipt.exists():
        raise RuntimeError("Detector run did not publish its receipt.")
print("FOUR_MASKRCNN_FOLDS_READY")

## 4. Reconstruct the eight-view consensus and H6 no-site crops

Only stage predictions and the SHA-256-verified TRAIN-trained selector feed the decision rule. The archived historical detector/H3 caches are never inputs. All 180 generated crops are saved separately for recoverability.

In [ ]:
from official_consensus import consensus
from p95_nosite_jpeg_crop_audit import rebuild_one

overlay_dets = OVERLAY / "dets_official.pkl"
OVERLAY.mkdir(parents=True, exist_ok=True)
for fold in FOLDS:
    link_dir = OVERLAY / "h3/maps" / f"{fold}_resnet34_s0"
    link_dir.mkdir(parents=True, exist_ok=True)
    link = link_dir / "official"
    if not link.is_symlink():
        if link.exists():
            raise RuntimeError(f"Unexpected path where an H3 link is required: {link}")
        link.symlink_to(H3 / fold, target_is_directory=True)
    if link.resolve() != (H3 / fold).resolve():
        raise RuntimeError("H3 overlay target changed.")

if not overlay_dets.exists():
    aggregates = {}
    for i, image in enumerate(FILES, 1):
        views = []
        for fold in FOLDS:
            with (DETECTORS / fold / f"{image.stem}.pkl").open("rb") as source:
                pairs = pickle.load(source)
            if len(pairs) != 2:
                raise RuntimeError(f"Expected two detector views: {image.name}")
            views.extend(pairs)
        aggregates[image.name] = consensus(views)
        if i % 30 == 0:
            print("CONSENSUS", i, "/180")
    temporary = overlay_dets.with_suffix(".partial")
    with temporary.open("xb") as sink:
        pickle.dump(aggregates, sink, protocol=pickle.HIGHEST_PROTOCOL)
        sink.flush()
        os.fsync(sink.fileno())
    os.replace(temporary, overlay_dets)
else:
    print("CONSENSUS_REUSED", sha(overlay_dets))

os.environ["SOLARFIL_ASSET_ROOT"] = str(OVERLAY)
# The historical H9b components() uses this explicit asset root.
import h9_official
if h9_official.O.resolve() != OVERLAY:
    raise RuntimeError("H9b model was imported against a different asset root.")
with selector_path.open("rb") as source:
    selector = pickle.load(source)

crop_receipts = OUT / "crop_receipts"
crop_receipts.mkdir(exist_ok=True)
for i, image in enumerate(FILES, 1):
    target = CROPS / f"{image.stem}.npz"
    evidence = crop_receipts / f"{image.stem}.json"
    if evidence.exists():
        r = json.loads(evidence.read_text(encoding="utf8"))
        if r.get("jpeg_sha256") != sha(image) or r.get("crop_sha256") != sha(target):
            raise RuntimeError(f"Resume crop checksum invalid: {image.name}")
        continue
    if target.exists():
        raise RuntimeError(f"Unattested crop output requires review: {target}")
    result = rebuild_one(image, selector, selector_meta)
    temporary = target.with_suffix(".partial")
    with temporary.open("xb") as sink:
        np.savez_compressed(sink, **result)
        sink.flush()
        os.fsync(sink.fileno())
    os.replace(temporary, target)
    write_json_once(evidence, {"image": image.name,
                              "jpeg_sha256": sha(image),
                              "crop_sha256": sha(target),
                              "instances": int(len(result["comp"]))})
    if i % 20 == 0:
        print("CROPS", i, "/180")
assert len(list(CROPS.glob("*.npz"))) == 180
print("ALL_PUBLIC_CROPS_READY")

## 5. Apply the two frozen crop refiners in CPU FP32

Both own pretrained checkpoints are used in the original four-flip inference scheme. No second CUDA process is launched. This stage is independent from Kaggle evaluation and can run while GPU1 hosts other work.

In [ ]:
refiner_files = []
cpu_env = os.environ.copy()
cpu_env["CUDA_VISIBLE_DEVICES"] = ""
cpu_env["OMP_NUM_THREADS"] = "2"
cpu_env["OPENBLAS_NUM_THREADS"] = "2"
cpu_env["MKL_NUM_THREADS"] = "2"
for encoder, checkpoint in (
    ("resnet34", "refiner_ALL.pt"),
    ("efficientnet_v2_s", "refiner_ALL_efficientnet_v2_s.pt"),
):
    result_file = REFINERS / f"{encoder}_probabilities.npz"
    if not result_file.exists():
        subprocess.run([
            sys.executable, "-u", str(SOURCES / "h9_infer.py"),
            "--weights", str(MODELS / checkpoint),
            "--encoder", encoder,
            "--crops", str(CROPS),
            "--out", str(result_file),
            "--device", "cpu",
        ], cwd=ROOT, env=cpu_env, check=True)
    with np.load(result_file, allow_pickle=False) as p:
        if set(p.files) != {f.stem for f in FILES}:
            raise RuntimeError(f"Refiner result incomplete: {encoder}")
    refiner_files.append(result_file)
    print("REFINER_VERIFIED", encoder, sha(result_file))

## 6. Compose disjoint instances and a valid COCO-RLE Kaggle CSV

The original H9b mask geometry, stable confidence order and two-model probability average are retained. The only deliberate deviation is the **public-only no-component fallback** described at the top; its frequency is included in the receipt. The notebook never accesses Kaggle hidden labels or submits any candidate.

In [ ]:
from filseg.postprocess import rle_counts
import h9_score

compiled_csv = OUT / "submission_public_rebuilt.csv"
compiled_receipt = OUT / "submission_public_rebuilt_receipt.json"
if compiled_receipt.exists():
    prior = json.loads(compiled_receipt.read_text(encoding="utf8"))
    if sha(compiled_csv) != prior["csv_sha256"]:
        raise RuntimeError("Previously completed CSV changed unexpectedly.")
    print("ALREADY_COMPLETED_AND_CHECKED", compiled_csv, prior["csv_sha256"])
else:
    if compiled_csv.exists():
        raise RuntimeError("An unattested output CSV exists; audit it before retrying.")
    pfiles = [np.load(path, allow_pickle=False) for path in refiner_files]
    fallback = []
    allrows = []
    try:
        for i, image in enumerate(FILES, 1):
            stem = image.stem
            with np.load(CROPS / f"{stem}.npz", allow_pickle=False) as archive:
                comp, scores, matrices = archive["comp"], archive["score"], archive["M"]
            prob, unet, mr, labels, areas, peaks = h9_official.components(
                image.name, selector_meta["w"], selector_meta["t"]
            )
            if len(comp):
                mean_probability = np.mean([
                    p[stem].astype(np.float32) for p in pfiles
                ], axis=0).round().astype(np.uint8)
                if mean_probability.shape[0] != len(comp):
                    raise RuntimeError(f"Refiner-crop misalignment: {stem}")
                seeds = [
                    h9_score.refined(labels == int(c) + 1,
                                     mean_probability[j], matrices[j])
                    for j, c in enumerate(comp)
                ]
                result = h9_score.assemble(
                    seeds, np.argsort(-scores, kind="stable")
                )
                masks = [rle_counts(result == v)
                         for v in range(1, int(result.max()) + 1)]
            else:
                # Public-only deterministic replacement for the private earlier
                # baseline. Never misrepresent this as exact historical H9b.
                fallback.append(stem)
                if len(areas):
                    pick = int(np.argmax(areas)) + 1
                    masks = [rle_counts(labels == pick)]
                else:
                    masks = [rle_counts(np.zeros((2048, 2048), dtype=bool))]
            allrows.extend((f"{stem}_{j}", rle)
                           for j, rle in enumerate(masks, 1))
            if i % 20 == 0:
                print("ASSEMBLED", i, "/180")
    finally:
        for p in pfiles:
            p.close()
    if len({ident for ident, _ in allrows}) != len(allrows):
        raise RuntimeError("The output has duplicate instance identifiers.")
    with compiled_csv.open("x", newline="", encoding="utf8") as sink:
        writer = csv.writer(sink, lineterminator="\n")
        writer.writerow(("filament_id", "segmentation_rle"))
        writer.writerows(allrows)
    receipt = {
        "status": "PUBLIC_RAW_IMAGE_REBUILT_NOT_OFFICIAL_SUBMISSION",
        "utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "images": len(FILES), "rows": len(allrows),
        "csv_sha256": sha(compiled_csv),
        "no_site_selector_sha256": sha(selector_path),
        "all_weights_sha256": manifest,
        "fallback_policy": "model_only_largest_semantic_component_or_empty",
        "fallback_images": fallback,
        "test_annotations_used": False,
        "official_kaggle_submission_made": False,
        "exact_historical_upload_parity_asserted": False,
    }
    write_json_once(compiled_receipt, receipt)
    print("PUBLIC_REPLAY_COMPLETED", compiled_csv, receipt["csv_sha256"],
          "rows", receipt["rows"], "fallback", len(fallback))

## 7. Independently validate COCO-RLE masks

The final integrity check decodes each RLE as a 2048 × 2048 COCO mask, verifies that no two submitted instances overlap, checks exact coverage of the 180 official image IDs, and preserves a write-once validation receipt. This is a schema and geometry audit, **not** a PQ measurement.

In [ ]:
from pycocotools import mask as mask_utils

checked = OUT / "geometry_validation_receipt.json"
if checked.exists():
    prev = json.loads(checked.read_text(encoding="utf8"))
    if prev["csv_sha256"] != sha(compiled_csv):
        raise RuntimeError("CSV changed after geometry audit.")
else:
    groups = {}
    with compiled_csv.open(newline="", encoding="utf8") as source:
        reader = csv.DictReader(source)
        if reader.fieldnames != ["filament_id", "segmentation_rle"]:
            raise RuntimeError("Wrong Kaggle CSV headers.")
        for entry in reader:
            stem = entry["filament_id"].rsplit("_", 1)[0]
            groups.setdefault(stem, []).append(entry["segmentation_rle"])
    if set(groups) != {p.stem for p in FILES}:
        raise RuntimeError("Submission does not cover exactly 180 test images.")
    checked_masks = 0
    for stem, masks in groups.items():
        occupied = np.zeros((2048, 2048), dtype=bool)
        for encoded in masks:
            binary = mask_utils.decode({
                "size": [2048, 2048], "counts": encoded.encode("ascii")
            }).astype(bool)
            if binary.shape != occupied.shape:
                raise RuntimeError(f"Wrong RLE dimensions for {stem}")
            if np.any(occupied & binary):
                raise RuntimeError(f"Overlapping instances in {stem}")
            occupied |= binary
            checked_masks += 1
    write_json_once(checked, {
        "status": "COCO_RLE_GEOMETRY_VALIDATED_NOT_PQ",
        "csv_sha256": sha(compiled_csv), "images": len(groups),
        "masks": checked_masks, "overlap_failures": 0,
        "hidden_or_test_labels_used": False,
    })
    print("GEOMETRY_VALIDATED", len(groups), checked_masks)

print("REPLAY_OUTPUT", compiled_csv)
print("REPLAY_DIGEST", sha(compiled_csv))

## Submission and scientific integrity

This notebook deliberately performs no Kaggle API upload. Its output must first pass a labelled **TRAIN-only** calibration and disjoint held-out validation, with independent comparisons against the best accepted H9b baseline. The public score **0.39 (submission 58112557)** is historical and must not be attributed to a newly rebuilt CSV. Do not optimize against the private leaderboard.

The final organiser Google Form, four-page English report, linked public repository, and confirmation receipt are separate mandatory delivery requirements. This experimental notebook must not be represented as the actual upload or as a verified clean-environment execution until its complete cells have run successfully on a new checkout.